In [72]:
import pandas as pd
import spacy

df = pd.read_csv("DataCollection/PostTokenizing/data.csv")
df = df.drop_duplicates()
df

,Title,Year,Genre,Description,Keywords
0,12 Years a Slave,2013,"Biography, Drama, History",Based on an incredible true story of one man's...,"['base', 'incredible', 'true', 'story', 'man',..."
1,13 Assassins,2010,"Action, Adventure, Drama","In 1844, the peace of Feudal Japan is threaten...","['1844', 'peace', 'feudal', 'japan', 'threaten..."
2,13 Hours,2014,"Short, Horror",Jack and his sister Jill are on the run after ...,"['jack', 'sister', 'jill', 'run', 'zombie', 'a..."
3,1917,2019,"Action, Drama, War","April 1917, the Western Front. Two British sol...","['april', '1917', 'western', 'british', 'soldi..."
4,300,2006,"Action, Drama",In the Battle of Thermopylae of 480 BC an alli...,"['battle', 'thermopylae', '480', 'bc', 'allian..."
...,...,...,...,...,...
269,X-Men: Days of Future Past,2014,"Action, Adventure, Sci-Fi","In the future, the mutants and the humans who ...","['future', 'mutant', 'human', 'help', 'slaught..."
270,X-Men: First Class,2011,"Action, Science Fiction, Adventure",Before Charles Xavier and Erik Lensherr took t...,"['charles', 'xavier', 'erik', 'lensherr', 'tak..."
271,X-Men: The Last Stand,2006,"Action, Adventure, Sci-Fi","When a ""cure"" is created, which apparently can...","['cure', 'create', 'apparently', 'turn', 'muta..."
272,X2: X-Men United,2003,"Action, Sci-Fi, Thriller",Several months have passed since the X-Men def...,"['month', 'pass', 'x', 'man', 'defeat', 'magne..."


In [73]:
import DataProcessing.tokenizing as tokenizing


In [74]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [75]:
# === Step 2: Define input and output ===
X = df[["Description", "Keywords"]]
y = df["Title"]

# === Step 3: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [76]:
# === Step 4: ColumnTransformer for multi-input text ===
preprocessor = ColumnTransformer(
    transformers=[
        ("desc_tfidf", TfidfVectorizer(max_features=1000, ngram_range=(1, 2), stop_words='english'), "Description"),
        ("kw_tfidf", TfidfVectorizer(max_features=500, ngram_range=(1, 2), stop_words='english'), "Keywords")
    ]
)

In [77]:
# === Step 5: Classifier pipeline ===
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [78]:
# === Step 6: Fit the model ===
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('desc_tfidf',
                                                  TfidfVectorizer(max_features=1000,
                                                                  ngram_range=(1,
                                                                               2),
                                                                  stop_words='english'),
                                                  'Description'),
                                                 ('kw_tfidf',
                                                  TfidfVectorizer(max_features=500,
                                                                  ngram_range=(1,
                                                                               2),
                                                                  stop_words='english'),
                                                  'Keywords')])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [79]:
# === Step 7: Evaluate ===
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.019230769230769232
                                            precision    recall  f1-score   support

                          A Beautiful Mind       0.00      0.00      0.00         1
                     A Man for All Seasons       0.00      0.00      0.00         0
            All Quiet on the Western Front       0.00      0.00      0.00         1
                           American Sniper       0.00      0.00      0.00         1
                           Angels & Demons       0.00      0.00      0.00         0
                                    Avatar       0.00      0.00      0.00         0
                  Avatar: The Way of Water       0.00      0.00      0.00         1
                   Avengers: Age of Ultron       0.00      0.00      0.00         1
                         Avengers: Endgame       0.00      0.00      0.00         0
                    Avengers: Infinity War       0.00      0.00      0.00         1
                               Baby Driver  

C:\Users\ibrah\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ibrah\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ibrah\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

In [ ]:
# === Step 8: Predict new sample ===
def predict_title(description):
    input_df = pd.DataFrame({"Description": [description], "Keywords": [" ".join(tokenizing.get_keywords(description))]})
    return pipeline.predict(input_df)[0]

Text = """Britain's Prince Albert (Colin Firth) must ascend the throne as King George VI, but he has a"""
# Example usage
print(predict_title(Text))

print(predict_title("A group of 300 Greek soldiers stand against the persian invasion of Greece"))

A Man for All Seasons
Dunkirk
